# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets along with their `@id` and fields
print("Available Record Sets:")
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- Record Set Name: {record_set.name}, @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name}, @id: {field.id}, type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
    print()

# If more details are needed for the first record set's fields/columns, print them separately
if dataset.record_sets:
    first_record_set = dataset.record_sets[0]
    print(f"Fields in record set '{first_record_set.name}':")
    for field in first_record_set.fields:
        print(f"  - {field.name} (@id: {field.id}), type: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by `@id`
dataframes = {}
for record_set in dataset.record_sets:
    print(f"Loading records from RecordSet: {record_set.name} (@id: {record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records loaded.")

# Pick the primary record set that contains protein quantitative data for further analysis
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]  # (adjust if you want a different set)
    main_df = dataframes[main_record_set_id]
    print(f"Selected record set for analysis: {main_record_set_id}")
    print(main_df.head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# We'll auto-select numeric columns from the primary DataFrame for demonstration
if main_record_set_id is not None:
    numeric_cols = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Numeric Field Selected for EDA: {numeric_field}")

        # Example threshold for filtering
        threshold = main_df[numeric_field].mean() + main_df[numeric_field].std()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Select a possible grouping field (use string/object columns)
        group_fields = main_df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for col in group_fields:
            if main_df[col].nunique() < 10 and main_df[col].nunique() > 1:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for analysis.")
else:
    print("No main DataFrame to explore.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id is not None and numeric_cols:
    # Histogram of the selected numeric field
    main_df[numeric_field].hist(bins=30, edgecolor='k', alpha=0.8)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot
    plt.figure(figsize=(8, 4))
    main_df.boxplot(column=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    # If we have group_field, plot group means
    if 'group_field' in locals() and group_field:
        means = main_df.groupby(group_field)[numeric_field].mean()
        means.plot(kind='bar', title=f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset defined by a Croissant schema, using the `mlcroissant` library. We explored the record sets, loaded tabular data into DataFrames, filtered and normalized quantitative protein data, and created basic visualizations. This workflow can be extended for more advanced bioinformatics or proteomics analyses.

Be sure to cite the dataset and check the `@id`s for record sets, fields, and columns if you integrate this data into your pipelines or publications.